# Ej. 2 · Un TWAP por recompensas (3,5 puntos)

**Objetivo (enunciado):** modificar el sistema de recompensas para replicar el comportamiento de un TWAP, explicar estado y
recompensa con gráficos (parte 1) y ajustar y motivar la parametría (parte 2). Debe funcionar para distintos días, horas,
duraciones y tamaños de orden; basta con imágenes correctas en train.

### Qué cambia respecto al ej. 1

| Bloque | Cambio | Motivo | Decisión |
|---|---|---|---|
| `_compute_reward` | `Default` → `Tracking` (seguimiento del plan) | La recompensa por defecto no es predecible con [T, V] | D-2.2 |
| `gamma` | 0,99 → 0,9 | Horizonte efectivo ≈ separación entre compras | D-2.3 |
| `learn_after`, `update_target_after` | 64 → 8; 640 → 1000 | Con 64 hay ~8 actualizaciones por episodio | D-2.4 |
| Evaluación | Rejilla N × ρ × días | Demuestra el requisito de generalidad | D-0.3 |

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("."))   # ejecutar desde notebooks/
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from common import data as dt, rewards as rw, exploration as ex, protocol as pr, plots as pl
from common.env import BestExecutionEnv
from common.agents import DDQNAgent, TWAPAgent
pd.set_option("display.precision", 3)
%matplotlib inline

In [ ]:
ASSET = "REP"                      # D-0.2
DATA = dt.load(ASSET)
train = DATA["train"]
GRID = pr.make_grid(train, n_days=3, seed=0)   # rejilla fija de evaluación (D-0.3)
print(dt.summary(DATA)); print(len(GRID), "episodios de evaluación")

## Parte 1 · Estado y recompensa

**Estado:** el del LAB, s = [T, V], con T = 1 − t/N y V = 1 − e/n (D-2.1). Basta: la regla del TWAP es una frontera en ese
plano (comprar si V ≥ T), y no depende del mercado porque el ej. 2 no busca precio.

**Recompensa (D-2.2):** penaliza la distancia entre lo ejecutado y el plan TWAP tras cada paso, en títulos y acotada:

$$r_t = -\frac{\min\left(\left|e_{t+1} - \tfrac{(t+1)\,n}{N}\right|,\ c\right)}{c} \in [-1, 0],\qquad c = 5$$

No le dice al agente qué acción tomar, solo qué es «ir bien»: comprar de más o de menos aleja del plan. Es **local** y
**factible** (depende solo del estado y de la acción), a diferencia del slippage frente al VWAP global del ej. 1. Como control
se entrena también con `TWAPRule` (+1 si la acción coincide con la regla del TWAP, −1 si no), que sí le dice qué hacer.

In [ ]:
CFG = dict(gamma=0.9, alpha=1e-3, buffer_size=50_000, batch_size=64, hidden_neurons=32)
TRAIN = dict(nepisodes=300, learn_after=8, update_target_after=1000)

def run(reward, seed=0, cfg=CFG, train_kw=TRAIN, exploration=None, label=None):
    pr.set_seeds(seed)
    env = BestExecutionEnv(train, reward=reward, seed=seed)
    agent = DDQNAgent(env, seed=seed, **cfg)
    pr.fill_buffer(agent, env, 25)
    exploration = exploration or ex.LinearDecay(1.0, 0.05, nepisodes=train_kw["nepisodes"])
    ep, ev = pr.train(agent, env, train_kw["nepisodes"], exploration, train_kw["learn_after"], train_kw["update_target_after"],
                      eval_data=train, eval_grid=GRID, eval_every=25, eval_reward=reward, log_every=0, verbose=False)
    res = pr.evaluate(agent, train, GRID, reward=reward)
    return agent, ep, ev, res

In [ ]:
agent, ep, ev, res = run(rw.Tracking())
agent.save(pr.MODELS / "ej2_ddqn.pt"); pr.save_df(ep, "ej2_train"); pr.save_df(ev, "ej2_evals"); pr.save_df(res, "ej2_eval_train")
_, _, _, res_rule = run(rw.TWAPRule())
pd.DataFrame({"DDQN · seguimiento": pr.summarize(res), "DDQN · regla (control)": pr.summarize(res_rule),
              "TWAP determinista": pr.summarize(pr.load_df("00_twap_grid"))})

In [ ]:
pl.plot_training(ep, ev, title="Ej. 2 · recompensa de seguimiento", save="ej2_training");

### Episodios distintos de la rejilla (día, hora, N y n distintos)

In [ ]:
agent.set_trainable(False)
for i in (0, 13, 26):
    cfg = GRID[i % len(GRID)]
    e = BestExecutionEnv(train, seed=0); agent.env = e
    pr.run_episode(e, agent, cfg)
    pl.plot_results(e, title=f"Ej. 2 · {cfg['day']} · N={cfg['n_bins']}, n={cfg['vol_care']}", save=f"ej2_episodio_{i}")

In [ ]:
envs = []
for cfg in GRID:
    e = BestExecutionEnv(train, seed=0); agent.env = e; pr.run_episode(e, agent, cfg); envs.append(e)
pl.plot_trajectories(envs, title="Ej. 2 · los 27 episodios de la rejilla", save="ej2_trayectorias")
pl.plot_policy_map(agent, title="Ej. 2 · política aprendida", save="ej2_politica");

## Parte 2 · Parametría

**γ (D-2.3).** Una compra cambia el error de seguimiento hasta la siguiente compra, unos N/n ≈ 10 pasos (1 de cada 10 con
ρ = 0,1). Lo que importa es que el agente vea ese efecto: con γ = 0,9 una recompensa a 10 pasos pesa 0,9¹⁰ ≈ 0,35; con 0,99
pesa 0,90 pero los Q-valores suman ~500 pasos y la diferencia entre acciones queda enterrada en un valor grande.

| γ | peso a 10 pasos | horizonte efectivo 1/(1−γ) |
|---|---|---|
| 0,9 | 0,35 | 10 pasos |
| 0,95 | 0,60 | 20 pasos |
| 0,99 | 0,90 | 100 pasos |

**Buffer (D-2.5):** 50.000 experiencias ≈ 100 episodios de ~500 pasos, la regla de clase. **Frecuencia de aprendizaje (D-2.4):**
aprender cada 8 pasos en lugar de cada 64. Barridos con 3 semillas (se cachean en `results/`; `RUN_SWEEPS = True` los recalcula).

In [ ]:
RUN_SWEEPS = not (pr.RESULTS / "ej2_sweeps.csv").exists()
SWEEPS = {"gamma": [0.9, 0.95, 0.99], "learn_after": [8, 64], "buffer_size": [10_000, 50_000]}
if RUN_SWEEPS:
    rows = []
    for param, values in SWEEPS.items():
        for v in values:
            for seed in range(3):
                cfg, tk = dict(CFG), dict(TRAIN)
                (tk if param == "learn_after" else cfg)[param] = v
                _, _, ev_s, res_s = run(rw.Tracking(), seed=seed, cfg=cfg, train_kw=tk)
                rows.append({"param": param, "value": v, "seed": seed, **pr.summarize(res_s).to_dict(),
                             "first_ok": ev_s.episode[ev_s.max_err <= 2].min()})
                print(param, v, seed, round(rows[-1]["max_err_mean"], 2))
    pr.save_df(pd.DataFrame(rows), "ej2_sweeps")
sweeps = pr.load_df("ej2_sweeps")
sweeps.groupby(["param", "value"])[["max_err_mean", "completed_mean", "t_fin_mean", "first_ok"]].agg(["mean", "std"])

## Conclusión

Completar con las cifras de la ejecución: error máximo de seguimiento frente al TWAP determinista (≤ 1 título), episodios
hasta alcanzarlo y efecto de γ, del buffer y de la frecuencia de aprendizaje.